In [13]:
import numpy as np
import os

class extra_functions:
    label_map = None
    sequences = []
    labels = []
    def __init__(self, no_sequence, sequence_length, data_path, actions):
        self.no_sequence = no_sequence
        self.sequence_length = sequence_length
        self.data_path = data_path
        self.actions = actions
    
    def make_folders(self):
        data_path = os.path.join(self.data_path)
        actions = np.array(self.actions)
        for action in actions:
            for sequence in range(self.no_sequence):
                try: 
                    os.makedirs(os.path.join(self.data_path, action, str(sequence)))
                except:
                    pass
    def return_no_sequence(self):
        return self.no_sequence
    def return_sequence_length(self):
        return self.sequence_length
    def creating_label_map(self):
        self.label_map = {label:num for num, label in enumerate(self.actions)}
    def return_label_map(self):
        return self.label_map
    def concating_gesture_sequences(self, path, no_sequence, sequence_length, actions, label_map):
        sequences, labels = [],[]
        for action in actions:
            for sequence in range(no_sequence):
                window=[]
                for frame_num in range(sequence_length):
                    if frame_num == sequence_length:
                        break
                    else:
                        res = np.load(os.path.join(path, action, str(sequence), "{}.npy".format(frame_num)))
                        res = res.flatten()
                        #print(res.shape)
                        window.append(res)
                sequences.append(window)
                labels.append(label_map[action])
        return sequences, labels

In [14]:
import sys
sys.path.append("/mnt/Main Drive/Codes/Deep Learning/Gesture_control")
import cv2
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

In [15]:
folder_name = "my_data"
actions = ['Close_Palm', 'Open_Palm','Pinch','Swipe_Down','Swipe_Left','Swipe_Right','Swipe_Up']
frames = 30
videos = 40

In [16]:
folder_setup = extra_functions(no_sequence=videos, sequence_length=frames, data_path=folder_name, actions=actions)
# callling the make folders function'
# folder_setup.make_folders()

In [17]:
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import cv2
import time
import mediapipe as mp
import numpy as np
from mediapipe import solutions
from mediapipe.framework.formats import landmark_pb2
import numpy as np
import threading 
import os

class Handlandmarks:
    #gesture = []
    BaseOptions = mp.tasks.BaseOptions
    HandLandmarker = mp.tasks.vision.HandLandmarker
    HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions
    HandLandmarkerResult = mp.tasks.vision.HandLandmarkerResult
    VisionRunningMode = mp.tasks.vision.RunningMode
    mp_drawing = mp.solutions.drawing_utils
    mp_hands = mp.solutions.hands
    handpoints = mp.tasks.vision.HandLandmarkerResult
    
    
    def __init__(self, keypoints,  frame, timestamp):
        self.frame = frame
        self.timestamp = timestamp
        self.keypoints = keypoints
        
    

    def get_landmarks(self):
        options = self.HandLandmarkerOptions(
                base_options=self.BaseOptions(model_asset_path='hand_landmarker.task'),
                running_mode=self.VisionRunningMode.LIVE_STREAM,
                result_callback=self.__result_callback)
        with self.HandLandmarker.create_from_options(options) as landmarker:
            np_array = cv2.cvtColor(self.frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=np_array )
            landmarker.detect_async(mp_image, self.timestamp)
    
    
    def __result_callback(self, result: mp.tasks.vision.HandLandmarkerResult, output_image: mp.Image, timestamp_ms: int):
        self.handpoints = result
        if len(result.hand_world_landmarks) == 0 : 
            for i in range(21):
                landmark_values = np.array([0.0, 0.0, 0.0])
                self.keypoints.append(landmark_values)
        else:
            hand_landmarks_list = result.hand_world_landmarks
            for idx in range(len(hand_landmarks_list)):
                hand_landmarks = hand_landmarks_list[idx]
                for landmarks in hand_landmarks:
                    #print("x:",landmarks.x)
                    #print("y:",landmarks.y)
                    #print("z:",landmarks.z)
                    landmark_values = np.array([landmarks.x,landmarks.y, landmarks.z])
                    self.keypoints.append(landmark_values)

    def returngesture(self):
        #self.keypoints = self.
        return self.keypoints
    
    def returnhandmarks(self):
        return self.handpoints
    
    def markings(self, frame, detection_result):
        hand_landmarks_list = detection_result.hand_landmarks
        for idx in range(len(hand_landmarks_list)):
            hand_landmarks = hand_landmarks_list[idx]
            hand_landmarks_proto = landmark_pb2.NormalizedLandmarkList()
            hand_landmarks_proto.landmark.extend([
            landmark_pb2.NormalizedLandmark(x=landmark.x, y=landmark.y, z=landmark.z) for landmark in hand_landmarks])
            solutions.drawing_utils.draw_landmarks(
                frame,
                hand_landmarks_proto,
                solutions.hands.HAND_CONNECTIONS,
                solutions.drawing_styles.get_default_hand_landmarks_style(),
                solutions.drawing_styles.get_default_hand_connections_style())
        return frame
    

In [18]:
# landmarks = None
# s = "http://127.0.0.1:4747/video"
# s1 = 0
# cap = cv2.VideoCapture(s)
# timestamp = 0
# action = actions[1]

# for action in actions:
#     for sequence in range(videos):
#         for frame_num in range(frames):
#             keypoints=[]
#             ret, frame = cap.read()
#             handlandmarks = Handlandmarks(keypoints, frame, timestamp)
#             handlandmarks.get_landmarks()
#             handconnections = handlandmarks.returnhandmarks()
#             landmarks = handlandmarks.returngesture()
#             frame = handlandmarks.markings(frame, handconnections)
            
#             if frame_num == 0 :
#                 cv2.putText(frame, ' starting collection', ( 120, 200), cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 4, cv2.LINE_AA)
#                 cv2.putText(frame, ' collecting frames for {} video number {}'.format(action, sequence), ( 15, 12), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,0,255), 1, cv2.LINE_AA)
#                 cv2.imshow('frame', frame)
#                 cv2.waitKey(1000)
#             else:
#                 cv2.putText(frame, ' collecting frames for {} video number {}'.format(action, sequence), ( 15, 12), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,0,255), 1, cv2.LINE_AA)
#                 cv2.imshow('frame', frame)
            
#             #print(landmarks)
#             npy_path = os.path.join(folder_name, action, str(sequence), str(frame_num))
#             #print(npy_path)
#             np.save(npy_path, landmarks)

#             timestamp = timestamp + 1
#             if cv2.waitKey(1) & 0xFF == ord('q'):
#                 break
# cap.release()
# cv2.destroyAllWindows()

In [19]:
folder_setup.creating_label_map()
label_map = folder_setup.return_label_map()
label_map

{'Close_Palm': 0,
 'Open_Palm': 1,
 'Pinch': 2,
 'Swipe_Down': 3,
 'Swipe_Left': 4,
 'Swipe_Right': 5,
 'Swipe_Up': 6}

In [20]:
sequences, labels = folder_setup.concating_gesture_sequences(path=folder_name, no_sequence = videos, sequence_length = frames, actions = actions, label_map=label_map)


In [21]:
X = np.array(sequences)
y = to_categorical(labels).astype(int)
print("shape of X Dataset:",X.shape)
print("shape of y Dataset:",y.shape)

shape of X Dataset: (280, 30, 63)
shape of y Dataset: (280, 7)


In [22]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3)
print("shape of X train Dataset:",X_train.shape)
print("shape of X test Dataset:",X_test.shape)
print("shape of y train Dataset:",y_train.shape)
print("shape of y test Dataset:",y_test.shape)

shape of X train Dataset: (196, 30, 63)
shape of X test Dataset: (84, 30, 63)
shape of y train Dataset: (196, 7)
shape of y test Dataset: (84, 7)


In [23]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.callbacks import TensorBoard

log_dir = os.path.join("logs")
tb_callback = TensorBoard(log_dir=log_dir)

model = Sequential()
model.add(LSTM(1024,return_sequences=True, activation='relu', input_shape=(30,63)))
model.add(LSTM(512,return_sequences=True, activation='relu',))
model.add(LSTM(512,return_sequences=True, activation='relu'))
model.add(LSTM(128,return_sequences=True, activation='relu'))
model.add(LSTM(128,return_sequences=True, activation='relu',dropout=0.1))
model.add(LSTM(128,return_sequences=False, activation='relu',dropout=0.1))
model.add(Dense(128, activation='relu'))
model.add(Dense(32, activation='relu'))
model.add(Dense(16, activation='relu'))
model.add(Dense(np.array(actions).shape[0], activation='softmax'))    

model.compile(optimizer='Adam', loss='categorical_crossentropy', metrics=['categorical_accuracy'])
model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 lstm_6 (LSTM)               (None, 30, 1024)          4456448   
                                                                 
 lstm_7 (LSTM)               (None, 30, 512)           3147776   
                                                                 
 lstm_8 (LSTM)               (None, 30, 512)           2099200   
                                                                 
 lstm_9 (LSTM)               (None, 30, 128)           328192    
                                                                 
 lstm_10 (LSTM)              (None, 30, 128)           131584    
                                                                 
 lstm_11 (LSTM)              (None, 128)               131584    
                                                                 
 dense_4 (Dense)             (None, 128)              

In [24]:
model.fit(X_train, y_train, epochs=1000,steps_per_epoch = 20, callbacks=[tb_callback]) # you only need 300 epochs


Epoch 1/1000
20/20 [==============================] - 10s 122ms/step - loss: 1.9470 - categorical_accuracy: 0.1173
Epoch 2/1000
20/20 [==============================] - 2s 121ms/step - loss: 1.9462 - categorical_accuracy: 0.1480
Epoch 3/1000
20/20 [==============================] - 3s 125ms/step - loss: 1.9455 - categorical_accuracy: 0.1582
Epoch 4/1000
20/20 [==============================] - 2s 119ms/step - loss: 1.9453 - categorical_accuracy: 0.1582
Epoch 5/1000
20/20 [==============================] - 2s 121ms/step - loss: 1.9450 - categorical_accuracy: 0.1582
Epoch 6/1000
20/20 [==============================] - 2s 121ms/step - loss: 1.9453 - categorical_accuracy: 0.1582
Epoch 7/1000
20/20 [==============================] - 2s 121ms/step - loss: 1.9447 - categorical_accuracy: 0.1582
Epoch 8/1000
20/20 [==============================] - 2s 124ms/step - loss: 63.5471 - categorical_accuracy: 0.1684
Epoch 9/1000
20/20 [==============================] - 2s 119ms/step - loss: 3.1070 - c

KeyboardInterrupt: 

In [ ]:
sdfghjkl;

In [25]:
model.save('LSTM_Landmark_only_Model_2.h5')
# model.save('LSTM_Landmark_only_Model_2')


/home/neutrino/miniconda3/envs/Ml/lib/python3.11/site-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [26]:
from sklearn.metrics import multilabel_confusion_matrix, accuracy_score
y_hat = model.predict(X_test)
y_true = np.argmax(y_test, axis=1).tolist()
y_hat = np.argmax(y_hat, axis=1).tolist()
multilabel_confusion_matrix(y_true, y_hat)

3/3 [==============================] - 1s 23ms/step


array([[[73,  1],
        [ 2,  8]],

       [[72,  1],
        [ 6,  5]],

       [[61,  8],
        [ 8,  7]],

       [[69,  6],
        [ 5,  4]],

       [[64,  9],
        [ 2,  9]],

       [[64,  7],
        [ 8,  5]],

       [[61,  8],
        [ 9,  6]]])

In [27]:
model.load_weights("LSTM_Landmark_only_Model_2.h5")


In [ ]:
import time
p = AudioControlsClass()
sequence = []
threshold = 0.4
landmarks = None
predictions = []
gesture = 'Nothing'
timestamp=0
cap = cv2.VideoCapture(0)
while True:
    keypoints=[]
    ret, frame = cap.read()
    handlandmarks = Handlandmarks(keypoints, frame, timestamp)
    handlandmarks.get_landmarks()
    landmarks = handlandmarks.returngesture()
    landmarks = np.array(landmarks).flatten()
    sequence.append(landmarks)
    sequence = sequence[-frames:]
    if len(sequence) == frames:
        res = model.predict(np.expand_dims(sequence, axis=0))[0]
        if res[np.argmax(res)]> threshold:
            predictions.append(actions[np.argmax(res)])
            #gesture = predictions[-1]
            gesture = p.prediction_gesture_handler(predictions)
            if gesture == "gesture 1":
                p.increase_volume()
                gesture = "Volumn Up"
            elif gesture == "gesture 2":
                p.decrease_volume()
                gesture = "Volumn Down"
            elif gesture == "gesture 3":
                p.switch_to_previous_desktop() 
                gesture = "Switching to previous Tab"               
            elif gesture == "gesture 4":
                p.switch_to_previous_desktop() 
                gesture = "Switching to next Tab"
            elif gesture == "gesture 5":
                p.take_screenshot()
                gesture = "Take Screenshot"
            elif gesture == "gesture 6":
                p.trigger_file_explorer()
                gesture = "open file explorer"
            

    cv2.putText(frame, str(gesture), (3,30), cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,255),2,cv2.LINE_AA)
    handconnections = handlandmarks.returnhandmarks()
    frame = handlandmarks.markings(frame, handconnections)
    cv2.imshow('frame', frame)
    timestamp = timestamp + 1
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

NameError: name 'AudioControlsClass' is not defined

In [29]:
import time
sequence = []
threshold = 0.4
landmarks = None
predictions = []
gesture = 'Nothing'
timestamp=0
cap = cv2.VideoCapture(0)
while True:
    keypoints=[]
    ret, frame = cap.read()
    handlandmarks = Handlandmarks(keypoints, frame, timestamp)
    handlandmarks.get_landmarks()
    landmarks = handlandmarks.returngesture()
    landmarks = np.array(landmarks).flatten()
    sequence.append(landmarks)
    sequence = sequence[-frames:]
    if len(sequence) == frames:
        res = model.predict(np.expand_dims(sequence, axis=0))[0]
        if res[np.argmax(res)]> threshold:
            predictions.append(actions[np.argmax(res)])
            gesture = predictions[-1]
            print(gesture) # Just print the gesture

    cv2.putText(frame, str(gesture), (3,30), cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,255),2,cv2.LINE_AA)
    handconnections = handlandmarks.returnhandmarks()
    frame = handlandmarks.markings(frame, handconnections)
    cv2.imshow('frame', frame)
    timestamp = timestamp + 1
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

I0000 00:00:1719227982.603362    6265 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1719227982.604536   15876 gl_context.cc:357] GL version: 3.2 (OpenGL ES 3.2 Mesa 23.2.1-1ubuntu3.1), renderer: AMD Radeon Graphics (renoir, LLVM 15.0.7, DRM 3.54, 6.5.0-28-generic)
I0000 00:00:1719227982.764476    6265 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1719227982.765448   15896 gl_context.cc:357] GL version: 3.2 (OpenGL ES 3.2 Mesa 23.2.1-1ubuntu3.1), renderer: AMD Radeon Graphics (renoir, LLVM 15.0.7, DRM 3.54, 6.5.0-28-generic)
I0000 00:00:1719227982.872516    6265 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1719227982.873613   15924 gl_context.cc:357] GL version: 3.2 (OpenGL ES 3.2 Mesa 23.2.1-1ubuntu3.1), renderer: AMD Radeon Graphics (renoir, LLVM 15.0.7, DRM 3.54, 6.5.0-28-generic)
I0000 00:00:1719227982.976733    6265 gl_context_egl.cc:85] Successfully initialized EGL. Majo

1/1 [==============================] - 0s 53ms/step


I0000 00:00:1719227984.913470    6265 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1719227984.914304   16444 gl_context.cc:357] GL version: 3.2 (OpenGL ES 3.2 Mesa 23.2.1-1ubuntu3.1), renderer: AMD Radeon Graphics (renoir, LLVM 15.0.7, DRM 3.54, 6.5.0-28-generic)
I0000 00:00:1719227985.069437    6265 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1719227985.070604   16483 gl_context.cc:357] GL version: 3.2 (OpenGL ES 3.2 Mesa 23.2.1-1ubuntu3.1), renderer: AMD Radeon Graphics (renoir, LLVM 15.0.7, DRM 3.54, 6.5.0-28-generic)


Close_Palm
1/1 [==============================] - 0s 31ms/step
Close_Palm
1/1 [==============================] - 0s 28ms/step
Close_Palm


I0000 00:00:1719227985.205234    6265 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1719227985.205981   16522 gl_context.cc:357] GL version: 3.2 (OpenGL ES 3.2 Mesa 23.2.1-1ubuntu3.1), renderer: AMD Radeon Graphics (renoir, LLVM 15.0.7, DRM 3.54, 6.5.0-28-generic)
I0000 00:00:1719227985.342324    6265 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1719227985.343494   16561 gl_context.cc:357] GL version: 3.2 (OpenGL ES 3.2 Mesa 23.2.1-1ubuntu3.1), renderer: AMD Radeon Graphics (renoir, LLVM 15.0.7, DRM 3.54, 6.5.0-28-generic)


...

I0000 00:00:1719228055.408463    6265 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1719228055.409454   37136 gl_context.cc:357] GL version: 3.2 (OpenGL ES 3.2 Mesa 23.2.1-1ubuntu3.1), renderer: AMD Radeon Graphics (renoir, LLVM 15.0.7, DRM 3.54, 6.5.0-28-generic)
I0000 00:00:1719228055.518297    6265 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1719228055.519311   37175 gl_context.cc:357] GL version: 3.2 (OpenGL ES 3.2 Mesa 23.2.1-1ubuntu3.1), renderer: AMD Radeon Graphics (renoir, LLVM 15.0.7, DRM 3.54, 6.5.0-28-generic)


Open_Palm
1/1 [==============================] - 0s 30ms/step
Open_Palm
1/1 [==============================] - 0s 28ms/step


I0000 00:00:1719228055.628147    6265 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1719228055.628976   37214 gl_context.cc:357] GL version: 3.2 (OpenGL ES 3.2 Mesa 23.2.1-1ubuntu3.1), renderer: AMD Radeon Graphics (renoir, LLVM 15.0.7, DRM 3.54, 6.5.0-28-generic)
I0000 00:00:1719228055.737634    6265 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1719228055.738595   37253 gl_context.cc:357] GL version: 3.2 (OpenGL ES 3.2 Mesa 23.2.1-1ubuntu3.1), renderer: AMD Radeon Graphics (renoir, LLVM 15.0.7, DRM 3.54, 6.5.0-28-generic)


Close_Palm
1/1 [==============================] - 0s 40ms/step
Close_Palm


I0000 00:00:1719228055.849634    6265 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1719228055.850504   37292 gl_context.cc:357] GL version: 3.2 (OpenGL ES 3.2 Mesa 23.2.1-1ubuntu3.1), renderer: AMD Radeon Graphics (renoir, LLVM 15.0.7, DRM 3.54, 6.5.0-28-generic)
